# BRCA 恶性细胞转移参考图谱：逐步运行教程

本 Notebook 复现以下流程：严格恶性细胞准备 → scPoli/contrastiveVI 多尺度表征 → DCIS/Primary/器官转移参考图谱 → 整样本留出预测 → 基因稳定性验证 → 外部新数据投射。

> 所有统计评价都以患者或样本为单位。图谱是标签监督的描述性参考空间，不能把训练集分离度直接解释为临床预测能力。

## 0. 环境与运行开关
第一次使用请先在终端运行 `conda env create -f environment.yml`，然后选择 `brca-metastasis-atlas` 内核。

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import anndata as ad
from IPython.display import display, Image, Markdown

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPTS = REPO / 'scripts'
ATLAS_ROOT = Path(os.environ.get('BRCA_ATLAS_ROOT', '/data/Home/liyuhan_codex/PanCanCer_process/fastCNV/BRCA_data_process/scimilarity_research/multiscale_metastasis_atlas_v1'))
os.environ['BRCA_ATLAS_ROOT'] = str(ATLAS_ROOT)
os.environ.setdefault('BRCA_DEVICE', 'cuda:0')

# 耗时步骤默认关闭。确认路径和GPU后再改为 True。
RUN_ATLAS_TRAINING = False
RUN_RISK_MODEL_TRAINING = False
RUN_GENE_ANALYSIS = False
RUN_EXTERNAL_MAPPING = False
print('Repository:', REPO)
print('Output root:', ATLAS_ROOT)
print('Python:', sys.executable)

## 1. 检查输入文件
`balanced_malignant_counts_4k.h5ad` 是样本平衡的训练矩阵；全量模型还会使用严格筛选后的120K细胞缓存。

In [ ]:
required = {
    'balanced atlas': ATLAS_ROOT / 'balanced_malignant_counts_4k.h5ad',
    'full malignant cache': ATLAS_ROOT / 'full_malignant_v4/full_strict_malignant_4k.h5ad',
}
for name, path in required.items():
    print(f'{name:24s}', 'OK' if path.exists() else 'MISSING', path)
assert required['balanced atlas'].exists(), '缺少 balanced_malignant_counts_4k.h5ad'
adata = ad.read_h5ad(required['balanced atlas'], backed='r')
print('Shape:', adata.shape)
display(pd.crosstab(adata.obs['disease_stage'], adata.obs['tissue_site']))
display(adata.obs.groupby(['GEO','disease_stage'], observed=True)['GSM'].nunique().unstack(fill_value=0))

## 2. 工具函数
每一步都调用仓库中的正式脚本，Notebook只负责组织、检查和展示结果。

In [ ]:
def run_script(name):
    path = SCRIPTS / name
    assert path.exists(), f'找不到脚本: {path}'
    print('Running:', path.name)
    subprocess.run([sys.executable, str(path)], check=True, cwd=SCRIPTS, env=os.environ.copy())

def show_json(path):
    path = Path(path)
    if path.exists():
        display(pd.json_normalize(json.loads(path.read_text(encoding='utf-8'))).T)
    else:
        print('Not generated:', path)

## 3. 训练多尺度参考组件

1. scPoli 显式建模研究/批次；
2. contrastiveVI 学习 General、Brain、Liver、Bone、Lymph-node 显著空间；
3. 阶段模型学习 DCIS→Primary、Primary→Metastasis 和 distant spread。

In [ ]:
if RUN_ATLAS_TRAINING:
    run_script('train_multiscale_reference.py')
    run_script('hierarchical_gene_screen.py')
else:
    print('Skipped. Set RUN_ATLAS_TRAINING=True to train the generative components.')

## 4. 构建 DCIS—Primary—器官转移参考图谱
该步骤融合多尺度隐空间，使用样本平衡抽样和研究对抗头训练六类参考坐标。图中每个点仍代表一个细胞。

In [ ]:
if RUN_ATLAS_TRAINING:
    run_script('train_stage_organ_map_with_dcis.py')
    run_script('build_multiscale_prototypes.py')
atlas_png = ATLAS_ROOT / 'stage_organ_single_cell_map_with_DCIS.png'
if atlas_png.exists(): display(Image(filename=str(atlas_png), width=900))
show_json(ATLAS_ROOT / 'stage_organ_map_with_DCIS_metrics.json')

### 图谱有效性检查

重点查看：每类细胞数、训练平衡准确率、类别 silhouette、研究 silhouette、混淆矩阵及原型支持的研究数。训练准确率只验证优化是否成功，真正的泛化证据来自整样本或整研究留出。

In [ ]:
proto = ATLAS_ROOT / 'organ_state_prototypes.csv'
if proto.exists():
    p = pd.read_csv(proto)
    display(p)
    display(p.groupby('organ').agg(n_prototypes=('prototype','size'), samples=('n_samples','sum'), cross_study=('eligible_cross_study','sum')))
show_json(ATLAS_ROOT / 'diagnostic_loso.json')

## 5. 训练全量转移风险模型
使用120,063个严格恶性细胞，同时保持整个GSM只出现在训练集或验证集的一侧。

In [ ]:
if RUN_RISK_MODEL_TRAINING:
    run_script('train_full_balanced_split_v4b.py')
else:
    print('Skipped model training; existing checkpoint will be used for evaluation.')
show_json(ATLAS_ROOT / 'full_balanced_split_v4b/metrics.json')

## 6. 整样本留出评估
同一个留出GSM的所有细胞都未参加编码器和分类头训练。细胞概率取样本均值后再判断Primary或Metastasis。

In [ ]:
run_script('evaluate_heldout_samples_v4b.py')
pred_path = ATLAS_ROOT / 'full_balanced_split_v4b/heldout_sample_predictions.csv'
pred = pd.read_csv(pred_path)
display(pred[['GSM','GEO','n_cells','true_stage','p_metastasis_mean','pred_stage','stage_correct','true_organ','pred_organ','organ_correct']])
display(Image(filename=str(ATLAS_ROOT / 'full_balanced_split_v4b/heldout_sample_predictions.png'), width=1000))

## 7. 基因归因与稳定性验证
依次运行积分梯度、样本伪bulk、跨研究效应、bootstrap、置换检验、CNV关联和TCGA证据整合。

In [ ]:
if RUN_GENE_ANALYSIS:
    for script in ['explain_full_v4b.py','validate_gene_stability_v4b.py','refine_gene_candidates_v4b.py','integrate_gene_evidence_v4b.py']:
        run_script(script)
gene_table = ATLAS_ROOT / 'full_balanced_split_v4b/gene_validation/integrated_gene_evidence.csv'
if gene_table.exists():
    genes = pd.read_csv(gene_table)
    display(genes[['integrated_rank','gene','evidence_class','IG_rank','paired_direction_consistency','bootstrap_positive_probability','cross_organ_positive_fraction','cnv_spearman','odds_ratio','p_value','fdr']].head(30))
heatmap = ATLAS_ROOT / 'full_balanced_split_v4b/gene_validation/gene_evidence_heatmap.png'
if heatmap.exists(): display(Image(filename=str(heatmap), width=1000))

## 8. 外部新数据投射
GSE158399脚本演示冻结参考图谱映射：QC→上皮候选→scPoli query surgery→contrastiveVI投影→原型/KNN/分类头一致性→unknown拒绝。新队列应复制此模板并更换输入读取部分。

In [ ]:
if RUN_EXTERNAL_MAPPING:
    run_script('external_validate_gse158399.py')
external_root = ATLAS_ROOT.parent / 'external_validation_GSE158399/results'
show_json(external_root / 'external_validation_metrics.json')
external_png = external_root / 'GSE158399_external_mapping.png'
if external_png.exists(): display(Image(filename=str(external_png), width=1000))

## 9. 结果解释边界

- 参考图谱的清晰分离证明模型能组织已知状态，不等于能预测未来转移。
- 新患者应用必须使用冻结模型，不能重新拟合标签。
- 主要指标必须按患者/样本报告，不能把细胞当成独立重复。
- 器官标签与取样组织天然相关；器官特异结果应称为转移灶适应状态，除非有同器官正常/非转移对照。
- 基因必须通过跨研究方向、bootstrap、置换、CNV/谱系污染、TCGA及独立队列验证后才能升级证据等级。